# Task 1: Classification

In this task we have trained a CNN model to lear to classify galaxy images according to the most voted answer in the first question of the kaggle galaxy zoo survey: whether a galaxy is smooth, has a disk or if the image is an artifact (not a galaxy).

This dataset presents an incredible class imbalance due to the very low number of artifacts (approximately 60), so we have taken 2 different approaches: multilabel classification applying inverse frequency weights to the loss function and binary classification, completely discarding the artifacts as they represent less than 1% of the dataset. The different architechture ("classification" for multilabel or "binary_classification") can be introduced to the different functions as an input.

To improve performance, an optional "tolerance" input has been included to discard any image where the vote consensus didn't reach the given tolerance in the survey (for example if less than 65% of the votes didn't agree, discard that image and only use data from images where a consensus was reached with our given tolerance).

In [ ]:
import torch
import numpy as np
from torch import nn
from torch.optim import lr_scheduler, AdamW
from torch.amp import GradScaler
from pathlib import Path

from utils.data import classification_dataloader
from utils.model import CNN
from utils.train import fit


ARCH = 'classification' # Options: "classification", "binary"
EPOCHS = 50
BATCH_SIZE = 64
IMG_SIZE = 128
LABELS_PATH = Path('data/labels.csv')


# Enables cudnn auto-tuner to find the best algorithm for the hardware
torch.backends.cudnn.benchmark = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


train_loader, val_loader, test_loader, train_data = classification_dataloader(
    labels_path=LABELS_PATH,
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    task=ARCH,
    tolerance=0.65 # Only keep confident crowdsourced votes
)


# Compute inverse frequency weights and apply square root scaling
# to prevent extreme weights from destabilizing the gradients.
n_classes = 3 if ARCH == 'classification' else 2
out_channels = 3 if ARCH == 'classification' else 1
class_counts = np.bincount(train_data['type'].values, minlength=n_classes)
raw_weights = 1.0 / (class_counts + 1e-8)
squashed_weights = np.sqrt(raw_weights)
final_weights = (squashed_weights / np.sum(squashed_weights)) * n_classes
weights_tensor = torch.tensor(final_weights, dtype=torch.float).to(device)

print(f'Training with class weights: {weights_tensor.cpu().numpy()}')


model = CNN(
    in_channels=3,
    out_channels=out_channels,
    input_shape=(IMG_SIZE, IMG_SIZE),
    hidden_channels=32,
    final_hidden_channels=512,
    n_blocks=5,
    architecture=ARCH
).to(device)


optimizer = AdamW(model.parameters(), lr=0.0003, weight_decay=1e-3)

# OneCycleLR smoothly ramps the learning rate up and down to prevent early plateauing
scheduler = lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=1e-3,
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS,
    pct_start=0.3,
    anneal_strategy='cos'
)

# GradScaler enables mixed precision training (fp16) for faster VRAM usage
scaler = GradScaler('cuda')

# Route to BCE for binary, CrossEntropy for multi-class
loss_fn = nn.BCEWithLogitsLoss(pos_weight=weights_tensor[1]) if ARCH == 'binary' else nn.CrossEntropyLoss(weight=weights_tensor)


train_losses, val_losses, train_acc, val_acc = fit(
    epochs=EPOCHS,
    loss_function=loss_fn,
    model=model,
    optimizer=optimizer,
    train_loader=train_loader,
    validation_loader=val_loader,
    device=device,
    scaler=scaler,
    architecture=ARCH,
    patience=6,
    scheduler=scheduler
)


## Testing

In this cell we run the tests for the trained model: plotting the training and validation losses as a function of the training epochs and running the model on the training dataset, obtaining the confusion matrix and ROC curves. For the test, we use the optimal tolerance computed from the validation dataset.

In [ ]:
from utils.plotting import plot_loss, execute_final_evaluation

plot_loss(train_losses, val_losses, train_acc, val_acc, task=ARCH)


model.load_state_dict(torch.load('best_model.pth'))
execute_final_evaluation(model, val_loader, test_loader, device)